In [1]:
#쿼드런트 : 온프레미스+클라우드환경
!pip install llama-index-vector-stores-qdrant==0.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [llama-index-vector-stores-qdrant]]


In [6]:
from llama_index.core import VectorStoreIndex
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from llama_index.core.schema import Document
from qdrant_client import QdrantClient

#메모리상에서만 실행
#client = QdrantClient(":memory:") 
#특정 디렉토리에 저장할 경우
client = QdrantClient(path="./qdrant_db")

#문서 데이터
documents = [
    "고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.",
    "강아지는 충성심이 강하고 친절한 동물로, 흔히 인간의 최고의 친구로 불립니다. 주로 애완동물로 기르고, 동반자로 유명합니다.",
    "고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다."
]
ids = ["doc1", "doc2", "doc3"]

#문서를 Document 형식으로 변환
documents = [Document(text=doc, id_=doc_id) for doc, doc_id in zip(documents, ids)]

#Qdrant에 새로운 컬렉션 생성
collection_name = "llama_index_qdrant"
client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=768, distance=Distance.COSINE)
)

#Qdrant 벡터 스토어 설정
vector_store = QdrantVectorStore(client=client, collection_name=collection_name)

#인덱스 생성 및 저장
index = VectorStoreIndex.from_documents(documents, vector_store=vector_store)

#쿼리 엔진 생성 및 실행
query_engine = index.as_query_engine()
query_text = "고양이"
response = query_engine.query(query_text)

#최종 응답 출력
print("[질의결과]")
print(response)

#응답 생성에 사용된 문서 확인
print("\n[응답에 사용된 문서]")
for i, node in enumerate(response.source_nodes, 1):
    print(f"{i}. {node.text}\n")

[질의결과]
고양이는 작은 육식동물로, 주로 애완동물로 기르며 민첩하고 장난기 있는 행동으로 유명합니다.

[응답에 사용된 문서]
1. 고양이는 작은 육식동물로, 주로 애완동물로 기릅니다. 민첩하고 장난기 있는 행동으로 유명합니다.

2. 고양이와 강아지는 전 세계적으로 인기 있는 애완동물로, 각각 독특한 특징을 가지고 있습니다.

